In [1]:
from pathlib import Path

# Resolve the checkout from either the project root or research/notebooks.
PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "run.py").is_file() and (p / "backend").is_dir()
)

import io
import json
import os
import requests
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [2]:
load_dotenv(PROJECT_ROOT / ".local/project.env")

PG_PARAMS = {
    "host":     os.getenv("DB_HOST", "localhost"),
    "port":     int(os.getenv("DB_PORT", "5432")),
    "user":     os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", ""),
    "dbname":   os.getenv("DB_NAME", "postgres"),
}

# SQLAlchemy pour DDL + substance_ref (petits volumes)
engine = create_engine(
    "postgresql+pg8000://",
    connect_args={
        "host":     PG_PARAMS["host"],
        "port":     PG_PARAMS["port"],
        "user":     PG_PARAMS["user"],
        "password": PG_PARAMS["password"],
        "database": PG_PARAMS["dbname"],
    },
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connexion OK")

Connexion OK


In [3]:
CSV_URL    = "https://pdh.cnrs.fr/download/full.csv"
CSV_LOCAL  = "full.csv"
CHUNK_SIZE = 20_000   # lignes par chunk — ajuster selon la RAM disponible

TABLE_POINT         = "point"
TABLE_SUBSTANCE_REF = "substance_ref"
TABLE_METADONNEE    = "metadonnee"

# date et dataset_id restent TEXT : formats hétérogènes dans le PDH
META_COLS = [
    "lat", "lon", "date", "year", "matrix", "unit",
    "category", "type", "sector", "country", "city", "name",
    "source_type", "data_collection_method",
    "source_text", "source_url",
    "dataset_id", "dataset_name", "pfas_sum",
]

In [4]:
print("Téléchargement du CSV PDH...")
response = requests.get(CSV_URL)
response.raise_for_status()

with open(CSV_LOCAL, "wb") as f:
    f.write(response.content)

# Compter les lignes une fois pour afficher la progression en passe 2
with open(CSV_LOCAL, "r", encoding="utf-8") as f:
    total_rows = sum(1 for _ in f) - 1  # -1 pour le header

n_chunks = (total_rows // CHUNK_SIZE) + 1
print(f"{total_rows:,} lignes — {n_chunks} chunks de {CHUNK_SIZE:,}.")

Téléchargement du CSV PDH...
622,698 lignes — 32 chunks de 20,000.


In [5]:
print("Passe 1 — collecte des substances...")

seen = {}  # key tuple -> substance_ref row

for chunk in pd.read_csv(
    CSV_LOCAL,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    usecols=["pfas_values"]
):
    for raw in chunk["pfas_values"]:
        if not isinstance(raw, str) or not raw.strip():
            continue

        try:
            measures = json.loads(raw)
            if not isinstance(measures, list):
                continue
        except (json.JSONDecodeError, ValueError, TypeError):
            continue

        for m in measures:
            if not isinstance(m, dict):
                continue

            substance_name = m.get("substance")
            cas_id = m.get("cas_id")
            isomer = m.get("isomer")
            unit_ref = m.get("unit")

            key = (
                str(substance_name).strip() if substance_name is not None else None,
                str(cas_id).strip() if cas_id is not None else None,
                str(isomer).strip() if isomer is not None else None,
                str(unit_ref).strip() if unit_ref is not None else None,
            )

            # ignorer les lignes vides
            if not key[0] and not key[1]:
                continue

            if key not in seen:
                seen[key] = {
                    "substance_name": key[0],
                    "cas_id": key[1],
                    "isomer": key[2],
                    "unit_ref": key[3],
                }

    del chunk  # libération explicite

df_substance_ref = (
    pd.DataFrame(seen.values())
    .sort_values(["substance_name", "cas_id", "isomer", "unit_ref"], na_position="last")
    .reset_index(drop=True)
)

df_substance_ref["pfas_index"] = df_substance_ref.index + 1

# Mapping clé composite -> nom de colonne pfas_N
substance_to_col = {
    (
        row["substance_name"],
        row["cas_id"],
        row["isomer"],
        row["unit_ref"],
    ): f"pfas_{row['pfas_index']}"
    for _, row in df_substance_ref.iterrows()
}

pfas_cols = [f"pfas_{i}" for i in range(1, len(df_substance_ref) + 1)]

print(f"Passe 1 OK : {len(df_substance_ref)} substances → pfas_1 … pfas_{len(df_substance_ref)}")

Passe 1 — collecte des substances...
Passe 1 OK : 547 substances → pfas_1 … pfas_547


In [6]:
pfas_col_ddl = ",\n".join(
    f"    pfas_{i} DOUBLE PRECISION" for i in range(1, len(df_substance_ref) + 1)
)

DDL = f"""
DROP TABLE IF EXISTS metadonnee CASCADE;
DROP TABLE IF EXISTS point CASCADE;
DROP TABLE IF EXISTS substance_ref CASCADE;

CREATE TABLE substance_ref (
    pfas_index      INTEGER PRIMARY KEY,
    substance_name  TEXT,
    cas_id          TEXT,
    isomer          TEXT,
    unit_ref        TEXT
);

CREATE TABLE point (
    point_id INTEGER PRIMARY KEY,
{pfas_col_ddl}
);

CREATE TABLE metadonnee (
    point_id                INTEGER PRIMARY KEY REFERENCES point(point_id),
    lat                     DOUBLE PRECISION,
    lon                     DOUBLE PRECISION,
    date                    DATE,
    year                    INTEGER,
    matrix                  TEXT,
    unit                    TEXT,
    category                TEXT,
    type                    TEXT,
    sector                  TEXT,
    country                 TEXT,
    city                    TEXT,
    name                    TEXT,
    source_type             TEXT,
    data_collection_method  TEXT,
    source_text             TEXT,
    source_url              TEXT,
    dataset_id              INTEGER,
    dataset_name            TEXT,
    pfas_sum                DOUBLE PRECISION,
    details_json            JSONB
);
"""

with engine.begin() as conn:
    for stmt in DDL.strip().split(";"):
        s = stmt.strip()
        if s:
            conn.execute(text(s))
print("Tables créées.")

Tables créées.


In [7]:
# substance_ref : ~88 lignes, to_sql suffit
df_substance_ref[["pfas_index", "substance_name", "cas_id", "isomer", "unit_ref"]].to_sql(
    TABLE_SUBSTANCE_REF, engine, if_exists="append", index=False
)
print(f"substance_ref chargée : {len(df_substance_ref)} lignes.")

substance_ref chargée : 547 lignes.


In [8]:
def copy_df(df: pd.DataFrame, table: str) -> None:
    """Charge un DataFrame dans PostgreSQL via COPY FROM STDIN.
    NaN → NULL. Ouvre et ferme sa propre connexion psycopg2.
    """
    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False, na_rep="\\N")
    buf.seek(0)
    cols = ", ".join(f'"{c}"' for c in df.columns)
    sql  = f'COPY "{table}" ({cols}) FROM STDIN WITH (FORMAT csv, NULL \'\\\\N\')'
    with psycopg2.connect(**PG_PARAMS) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(sql, buf)
        conn.commit()

In [9]:
import math
def safe_load_list(x):
    if not isinstance(x, str) or not x.strip():
        return []
    try:
        v = json.loads(x)
        return v if isinstance(v, list) else []
    except Exception:
        return []

def safe_load_dict(x):
    if not isinstance(x, str) or not x.strip():
        return {}
    try:
        v = json.loads(x)
        return v if isinstance(v, dict) else {}
    except Exception:
        return {}

def to_bool(x):
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "y"}
    return bool(x)

def clean_json_value(x):
    if isinstance(x, dict):
        return {k: clean_json_value(v) for k, v in x.items()}
    if isinstance(x, list):
        return [clean_json_value(v) for v in x]
    if isinstance(x, float):
        if math.isnan(x) or math.isinf(x):
            return None
    return x

def safe_json_dumps(obj):
    cleaned = clean_json_value(obj)
    return json.dumps(cleaned, ensure_ascii=False, allow_nan=False)

In [10]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE metadonnee CASCADE;"))
    conn.execute(text("TRUNCATE TABLE point CASCADE;"))

print("Tables point et metadonnee vidées.")

Tables point et metadonnee vidées.


In [11]:
print("Passe 2 — vectorisation + chargement...")

point_id_offset = 0
chunks_done = 0
total_loaded = 0

def copy_df(df: pd.DataFrame, table: str) -> None:
    """Charge un DataFrame dans PostgreSQL via COPY FROM STDIN.
    NaN -> NULL.
    """
    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False, na_rep="\\N")
    buf.seek(0)

    cols = ", ".join(f'"{c}"' for c in df.columns)

    sql = f"""COPY "{table}" ({cols})
              FROM STDIN
              WITH (FORMAT csv, NULL '\\N')"""

    with psycopg2.connect(**PG_PARAMS) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(sql, buf)
        conn.commit()

for chunk in pd.read_csv(CSV_LOCAL, chunksize=CHUNK_SIZE, low_memory=False):
    chunk = chunk.reset_index(drop=True)
    chunk_len = len(chunk)
    chunk["point_id"] = chunk.index + point_id_offset + 1

    # Parsing JSON sécurisé
    chunk["_pfas"] = chunk["pfas_values"].apply(safe_load_list)
    chunk["_det"] = chunk["details"].apply(safe_load_dict)

    # Construire les vecteurs point pour ce chunk uniquement
    point_records = []
    for row in chunk[["point_id", "_pfas"]].itertuples(index=False, name=None):
        point_id, pfas_list = row

        vec = dict.fromkeys(pfas_cols, None)
        vec["point_id"] = point_id

        for m in pfas_list:
            key = (
                str(m.get("substance")).strip() if m.get("substance") is not None else None,
                str(m.get("cas_id")).strip() if m.get("cas_id") is not None else None,
                str(m.get("isomer")).strip() if m.get("isomer") is not None else None,
                str(m.get("unit")).strip() if m.get("unit") is not None else None,
            )

            col = substance_to_col.get(key)
            if col:
                lt = to_bool(m.get("less_than"))
                vec[col] = None if lt else m.get("value")

        point_records.append(vec)

    df_point_chunk = pd.DataFrame(point_records)[["point_id"] + pfas_cols]

    # Conversion numérique explicite
    df_point_chunk[pfas_cols] = df_point_chunk[pfas_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    # Construire metadonnee
    available_meta = [c for c in META_COLS if c in chunk.columns]
    df_meta_chunk = chunk[["point_id"] + available_meta].copy()
    df_meta_chunk["details_json"] = chunk["_det"].apply(safe_json_dumps)

    # year -> entier nullable
    if "year" in df_meta_chunk.columns:
        df_meta_chunk["year"] = (
            pd.to_numeric(df_meta_chunk["year"], errors="coerce")
            .round(0)
            .astype("Int64")
        )

    # dataset_id -> entier nullable
    if "dataset_id" in df_meta_chunk.columns:
        df_meta_chunk["dataset_id"] = (
            pd.to_numeric(df_meta_chunk["dataset_id"], errors="coerce")
            .astype("Int64")
        )

    # convertir les Int64 pandas en object + None pour COPY
    for c in df_meta_chunk.columns:
        if str(df_meta_chunk[c].dtype) == "Int64":
            df_meta_chunk[c] = df_meta_chunk[c].astype(object).where(
                df_meta_chunk[c].notna(), None
            )

    # COPY : point avant metadonnee
    copy_df(df_point_chunk, TABLE_POINT)
    copy_df(df_meta_chunk, TABLE_METADONNEE)

    point_id_offset += chunk_len
    total_loaded += chunk_len
    chunks_done += 1

    del chunk, df_point_chunk, df_meta_chunk, point_records

    print(f"  [{chunks_done}/{n_chunks}] {total_loaded:,}/{total_rows:,} lignes chargées")

print(f"\nPasse 2 OK : {total_loaded:,} points chargés.")

Passe 2 — vectorisation + chargement...
  [1/32] 20,000/622,698 lignes chargées
  [2/32] 40,000/622,698 lignes chargées
  [3/32] 60,000/622,698 lignes chargées
  [4/32] 80,000/622,698 lignes chargées
  [5/32] 100,000/622,698 lignes chargées
  [6/32] 120,000/622,698 lignes chargées
  [7/32] 140,000/622,698 lignes chargées
  [8/32] 160,000/622,698 lignes chargées
  [9/32] 180,000/622,698 lignes chargées
  [10/32] 200,000/622,698 lignes chargées
  [11/32] 220,000/622,698 lignes chargées
  [12/32] 240,000/622,698 lignes chargées
  [13/32] 260,000/622,698 lignes chargées
  [14/32] 280,000/622,698 lignes chargées
  [15/32] 300,000/622,698 lignes chargées
  [16/32] 320,000/622,698 lignes chargées
  [17/32] 340,000/622,698 lignes chargées
  [18/32] 360,000/622,698 lignes chargées
  [19/32] 380,000/622,698 lignes chargées
  [20/32] 400,000/622,698 lignes chargées
  [21/32] 420,000/622,698 lignes chargées
  [22/32] 440,000/622,698 lignes chargées
  [23/32] 460,000/622,698 lignes chargées
  [24/3

In [12]:
INDEXES = """
CREATE INDEX IF NOT EXISTS idx_meta_country  ON metadonnee (country);
CREATE INDEX IF NOT EXISTS idx_meta_matrix   ON metadonnee (matrix);
CREATE INDEX IF NOT EXISTS idx_meta_year     ON metadonnee (year);
CREATE INDEX IF NOT EXISTS idx_meta_latlon   ON metadonnee (lat, lon);
CREATE INDEX IF NOT EXISTS idx_meta_dataset  ON metadonnee (dataset_id);
"""

with engine.begin() as conn:
    for stmt in INDEXES.strip().split(";"):
        s = stmt.strip()
        if s:
            conn.execute(text(s))

print("Index créés.")

Index créés.


In [13]:
with engine.connect() as conn:
    n_point = conn.execute(text("SELECT COUNT(*) FROM point")).scalar()
    n_sub = conn.execute(text("SELECT COUNT(*) FROM substance_ref")).scalar()
    n_meta = conn.execute(text("SELECT COUNT(*) FROM metadonnee")).scalar()

print(f"point         : {n_point:>10,} lignes")
print(f"substance_ref : {n_sub:>10,} substances")
print(f"metadonnee    : {n_meta:>10,} lignes")

assert n_point == n_meta, "Désalignement point / metadonnee !"
assert n_point == total_loaded, "Désalignement point / lignes CSV !"

print("\nConsistance OK — ETL V2 terminé.")

point         :    622,680 lignes
substance_ref :        547 substances
metadonnee    :    622,680 lignes

Consistance OK — ETL V2 terminé.
